# 작업 중에 들어온 오타 수정과 팀 리뷰

앞 단원에서 푼 문제 하나를 고르세요. 문법 과정에서는 값을 하나 바꿔 다시 실행한 예제, 패턴 과정에서는 같은 입력을 다른 기준으로 처리한 예제가 좋습니다. 다음 사람이 내 결과를 다시 낼 수 있도록 입력·기대값·실제값을 한 줄로 적습니다. 이 한 줄과 예제 셀이 이틀 동안 팀에서 검토할 자료입니다.

먼저 작성 중인 예제를 그대로 두고 별도 폴더에서 오타만 고칩니다. 그다음 짝이 내 변경을 읽고 새 입력을 제안합니다. 리뷰를 반영해 같은 요청을 갱신하고, 팀의 기본 버전에 두 사람의 변경을 합칩니다.

Python 3 커널과 Git이 필요합니다. Windows는 WSL 터미널에서 `git --version`을 실행하고 WSL에서 연 노트북을 사용하세요. macOS는 터미널에서 같은 명령을 씁니다. 버전이 나오면 다음 셀을 실행합니다. 명령을 찾지 못하면 [Git 설치 안내](https://git-scm.com/downloads)에서 해당 환경을 선택하세요. 아래 코드 셀은 임시 폴더만 사용하므로 혼자 반복할 수 있습니다. 두 사람의 GitHub 실습은 뒤쪽의 팀 과제와 [팀 활동지](practice.md)를 따릅니다.

## 커밋이 보관하는 변경 읽기

파일을 저장하면 지금 열어 둔 폴더의 내용이 달라집니다. Git은 **저장소**에 파일의 변경 이력을 보관합니다. 어떤 파일을 다음 기록에 넣을지 고르는 명령이 `git add`, 고른 내용을 설명과 함께 한 차례 기록하는 명령이 `git commit`입니다. 이렇게 남긴 기록 한 건을 **커밋**이라고 합니다. 먼저 준비 셀을 실행해 예제 파일 하나를 담은 저장소를 만드세요. 이 준비 셀은 수정하지 말고 한 번만 실행하세요. 임시 폴더를 만들고 그 안에서 Git을 실행하는 부분은 강사가 준비했습니다. 뒤에서 `git(repo, "status")`라고 쓰면 첫 폴더에서 터미널 명령 `git status`를 실행합니다. `git(fix, "status")`는 두 번째 폴더에서 같은 명령을 실행합니다. Python 셀에서는 따옴표로 인자를 나누고, 터미널에서는 공백으로 구분합니다.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import subprocess

# 새 임시 폴더 안에서만 명령을 실행합니다. 첫 셀부터 다시 실행하면 새 실습이 시작됩니다.
session = TemporaryDirectory()
root = Path(session.name)
repo = root / "team"
fix = root / "spelling"
repo.mkdir()

def git(folder, *args):
    # -C 뒤에 작업 폴더를 넘기면, 다른 저장소에서 명령을 실행할 염려가 없습니다.
    result = subprocess.run(["git", "-C", str(folder), *args], capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(result.stderr or result.stdout)
    return result.stdout.rstrip()

git(repo, "init", "-b", "main")  # main을 첫 브랜치 이름으로 정합니다.
git(repo, "config", "user.name", "Classroom")
git(repo, "config", "user.email", "classroom@example.invalid")
git(repo, "config", "commit.gpgsign", "false")  # 연습 저장소의 커밋에는 서명을 사용하지 않습니다.
guide = repo / "cases.md"
_ = guide.write_text("문벙 예제에서는 값을 하나 바꿔 보세요.\n", encoding="utf-8")
git(repo, "add", "cases.md")
git(repo, "commit", "-m", "첫 실습 안내 추가")  # -m 뒤에 이 커밋의 설명을 씁니다.
print(git(repo, "log", "-1", "--format=%s"))

첫 실습 안내 추가


`첫 실습 안내 추가`는 방금 저장한 커밋의 설명입니다. `log`는 기록을 읽고, `-1`은 가장 최근 한 건을 고르며, `--format=%s`는 제목만 보여 줍니다. 이후의 `git(repo, "status")`는 터미널에서 이 폴더를 열고 `git status`를 쓴 것과 같습니다.

### 예제를 쓰던 중 오타 제보가 들어오면?

**브랜치**는 이어서 커밋할 작업의 이름입니다. 팀의 기준 버전은 `main`에 두고, 내 예제는 새 브랜치에 작성합니다. `switch -c` 뒤에 이름을 적으면 현재 커밋에서 브랜치를 만들고 그곳으로 이동합니다.

In [2]:
git(repo, "switch", "-c", "case/min")
draft = repo / "min-case.md"
_ = draft.write_text("입력 1시간 2분 3초 / 기대값 3723초 / 실제값 3723초\n", encoding="utf-8")
print(git(repo, "status", "--short", "--branch"))  # 현재 브랜치와 변경 파일을 함께 봅니다.

## case/min
?? min-case.md


`## case/min`은 현재 브랜치입니다. 그 아래 `?? min-case.md`는 <mark>디스크에 파일이 생겼지만 Git이 아직 추적하지 않는다는 뜻입니다.</mark> 여기서 커밋하기에는 설명이 덜 쓰였습니다. 그런데 동료가 기준 버전의 ‘문벙’이라는 오타를 발견했습니다.

### 같은 저장소의 다른 작업 폴더 열기

**워크트리(worktree)**는 어느 브랜치의 파일을 실제로 펼쳐 놓은 작업 폴더입니다. 한 저장소에 두 번째 워크트리를 만들면 두 브랜치의 파일을 서로 다른 창에서 열 수 있습니다. 아래에서 `-b` 뒤는 새 브랜치 이름, 그다음은 새 폴더 경로, 마지막 `main`은 출발할 커밋을 가리킵니다.

In [3]:
git(repo, "worktree", "add", "-b", "fix/min-type", str(fix), "main")
print(git(fix, "status", "--short", "--branch"))

## fix/min-type


두 번째 폴더에는 `## fix/min-type`만 보입니다. 첫 폴더에서 작성 중인 파일은 아직 커밋에 들어 있지 않아서 `main`에서 연 두 번째 폴더에 없습니다. 이 폴더의 안내문만 고쳐 봅시다.

In [4]:
other_guide = fix / "cases.md"
before = other_guide.read_text(encoding="utf-8")
_ = other_guide.write_text(before.replace("문벙", "문법"), encoding="utf-8")
print(git(fix, "diff", "--", "cases.md"))  # 수정한 파일을 커밋하기 전에 비교합니다.

diff --git a/cases.md b/cases.md
index 178bb98..c207984 100644
--- a/cases.md
+++ b/cases.md
@@ -1 +1 @@
-문벙 예제에서는 값을 하나 바꿔 보세요.
+문법 예제에서는 값을 하나 바꿔 보세요.


파일 이름 아래에서 `-`로 시작하는 줄은 고치기 전 내용이고, `+`로 시작하는 줄은 고친 뒤 내용입니다. ‘문벙’이 ‘문법’으로 바뀌었는지 두 줄을 읽어 보세요. 이제 이 파일을 다음 커밋에 넣습니다.

In [5]:
git(fix, "add", "cases.md")
print(git(fix, "diff", "--cached", "--name-only"))  # 다음 커밋에 들어갈 파일 이름만 표시합니다.

cases.md


목록에는 `cases.md`가 한 번 나옵니다. 여기에 `min-case.md`까지 보인다면 지금 어느 폴더의 변경을 골랐는지 확인하세요.

In [6]:
git(fix, "commit", "-m", "문법 실습 안내의 오타 수정")
print(git(fix, "log", "-1", "--format=%s"))

문법 실습 안내의 오타 수정


`fix/min-type` 브랜치의 최신 기록은 오타 수정입니다. 첫 폴더에는 어떤 상태가 남았을까요?

### 직접 확인 · 커밋 제목과 브랜치 이름 구별하기

첫 폴더의 현재 브랜치와 미커밋 파일을 출력하세요. 다음 줄에는 두 번째 폴더의 최신 커밋에 들어간 파일 이름만 출력하세요. 최신 커밋을 가리키는 이름은 `HEAD`입니다. `git show --format= --name-only HEAD`에서 빈 `--format=`은 제목 등의 출력을 생략합니다.

In [7]:
# 여기에 확인 명령을 쓰고 실행하세요.

<details><summary>확인 명령과 실행 결과 보기</summary>
<pre><code># 첫 폴더에서 예제 초안이 그대로 있는지 확인합니다.
print(git(repo, &quot;status&quot;, &quot;--short&quot;, &quot;--branch&quot;))
# 둘째 폴더에서 방금 기록한 파일만 확인합니다.
print(git(fix, &quot;show&quot;, &quot;--format=&quot;, &quot;--name-only&quot;, &quot;HEAD&quot;))</code></pre><pre>## case/min
?? min-case.md
cases.md</pre><p>첫 폴더에는 작성 중인 min-case.md가 남아 있습니다. 둘째 폴더의 최신 커밋에는 cases.md만 들어 있습니다. 동료에게 오타 수정을 먼저 보낼 브랜치는 fix/min-type입니다.</p>
</details>

## 팀에 보내서 같은 PR에서 수정하기

지금까지의 커밋은 내 컴퓨터에만 있습니다. GitHub는 팀이 함께 사용할 저장소를 제공합니다. `clone`은 그 저장소와 이력을 내 컴퓨터에 복제합니다. 복제한 출처의 기본 별명은 `origin`입니다. `push`하면 내 브랜치에 남긴 커밋이 팀 저장소의 같은 브랜치에 올라갑니다. 팀의 `main`에 합쳐 달라는 요청은 **Pull Request(PR)**라고 합니다. 동료가 읽은 뒤 변경을 제안하는 과정이 리뷰입니다.

팀 실습은 둘이 한 조로 진행합니다. [시작 문서](cases.md)에는 각자 고칠 오타가 하나씩 있습니다. [팀 활동지](practice.md)에서 팀 저장소를 준비하고, 워크트리 실험을 자기 Python 예제로 다시 해 보세요. 이어서 오타와 예제를 각각 별도 PR로 보냅니다. 아래 질문을 먼저 풀고 실제 주소와 커밋을 활동지에 남기세요.

### 팀 과제 1 · 어느 브랜치에서 리뷰를 반영할까요?

짝이 내 예제를 읽고 “입력을 하나 바꾸면 어떤 결과가 나오는지 보여 주세요”라고 했습니다. 내 예제 PR이 열린 상태입니다. 앞 단원의 코드를 새 입력으로 실행하고 그 결과를 자기 예제 기록에 추가하세요. 어느 폴더에서 어떤 파일을 기록하고 어디로 보내야 할까요? 짝은 기존 PR의 파일 차이에서 새 입력과 출력을 찾아야 합니다.

<details><summary>민의 명령과 리뷰 답변 예시 보기</summary>
<pre><code># team 폴더에서 현재 브랜치가 case/min인지 확인합니다.
git branch --show-current
git add min/case.ipynb min/cases.md
git commit -m &quot;0초 입력을 실행하고 결과를 기록&quot;
git push origin case/min</code></pre><pre>입력 0시간 0분 0초 / 기대값 0초 / 실제값 0초</pre><p>학생이 팀에서 남길 기록의 예입니다. 같은 원격 브랜치에 올렸으므로 기존 PR에 수정 커밋이 추가됩니다. 짝에게 변경한 셀과 케이스를 짚어 답글을 남기세요. 실제 파일 이름과 값은 자신이 실행한 예제를 따릅니다.</p>
</details>

### 팀 과제 2 · 같은 안내 줄을 둘 다 고치면?

민과 윤이 같은 기본 커밋에서 루트 `cases.md`의 ‘확인할 것’ 한 줄을 다르게 고쳤습니다. 민의 변경이 먼저 병합되면 Git은 윤의 줄과 민의 줄을 자동으로 합칠 수 없습니다. 이 상태가 **충돌**입니다. 두 사람이 추가한 조건을 모두 살려 한 줄로 남기세요. 팀에서 일어난 충돌을 활동지 순서대로 해결한 뒤 원래 예제와 동료 예제를 다시 실행합니다.

<details><summary>충돌한 줄과 합친 결과 예시 보기</summary>
<pre><code>&lt;&lt;&lt;&lt;&lt;&lt;&lt; HEAD
확인할 것: 입력과 실제값, 실행 순서
=======
확인할 것: 입력과 실제값, 단위
&gt;&gt;&gt;&gt;&gt;&gt;&gt; origin/main</code></pre><pre>확인할 것: 입력과 실제값, 단위, 실행 순서</pre><p>윤이 자기 브랜치에서 origin/main을 합친 경우입니다. 위쪽에는 현재 브랜치의 줄, 아래쪽에는 가져온 줄이 보입니다. 두 요구를 한 줄로 쓰고 경계 표식 세 줄을 모두 지운 뒤 Git에 해결한 파일을 등록하세요.</p>
</details>

### 팀 과제 3 · 짝의 컴퓨터에서 다시 실행하기

양쪽 PR이 병합된 후 짝은 자신의 기본 브랜치를 갱신합니다. 내 예제 노트북을 새 커널로 열어 기록에 적힌 첫 입력과 리뷰 때 추가한 입력을 각각 실행해 보세요. 셀을 실행할 때 필요한 변수가 들어 있는지, 실제 출력이 예제 기록과 맞는지 확인하세요. 결과가 다르면 입력과 실제 출력을 PR에 남깁니다.

<details><summary>다시 실행한 동료의 기록 예시 보기</summary>
<pre><code># 갱신한 team 폴더에서 예제 셀의 파일과 마지막 변경을 읽습니다.
git log -1 --format=%s -- min/case.ipynb</code></pre><pre>0초 입력을 실행하고 결과를 기록</pre><p>실습용 커밋 제목의 예입니다. 이 제목만으로 셀의 실행을 확인할 수는 없습니다. 짝이 노트북을 직접 실행하고 ‘첫 입력 3723초, 새 입력 0초를 재현했습니다’처럼 자신이 본 값을 남겨야 합니다.</p>
</details>

## 다음 실습에서 가져갈 것

활동지에 자신이 올린 예제 PR, 동료에게 남긴 리뷰와 반영한 커밋을 기록하세요. 다음 단원에서는 모델에게 보낸 입력과 돌아온 응답을 구분합니다. 이번에 만든 팀 저장소에서 새 작업 브랜치를 여는 방법과, 동료가 다시 실행할 수 있도록 입력·기대값·실제값을 남긴 방식을 그대로 쓸 수 있습니다.